In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("Celebal Assignment") \
    .getOrCreate()

In [4]:
from google.colab import files
uploaded = files.upload()

Saving employees.csv to employees.csv


In [5]:
df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

In [6]:
df.show()

+---+-----+----+----------+------+------+
| ID| Name| Age|Department|Salary|Region|
+---+-----+----+----------+------+------+
|  1|Rahul|  22|        IT| 40000| North|
|  2| Aman|  25|        HR| 35000|  West|
|  3| Neha|  30|        IT| 60000| North|
|  4| Riya|NULL|   Finance| 45000| South|
|  5| Amit|  28|        HR|  NULL|  East|
|  6|Rahul|  22|        IT| 40000| North|
|  7|Ankit|  19|     Sales| 25000|  West|
|  8|Priya|  35|   Finance| 80000| South|
|  9| NULL|  26|        IT| 50000|  East|
| 10|Vikas|  40|     Sales| 90000|  NULL|
+---+-----+----+----------+------+------+



In [7]:
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Region: string (nullable = true)



In [8]:
df = df.dropDuplicates()

In [9]:
df.show()

+---+-----+----+----------+------+------+
| ID| Name| Age|Department|Salary|Region|
+---+-----+----+----------+------+------+
|  2| Aman|  25|        HR| 35000|  West|
|  3| Neha|  30|        IT| 60000| North|
|  7|Ankit|  19|     Sales| 25000|  West|
|  8|Priya|  35|   Finance| 80000| South|
|  6|Rahul|  22|        IT| 40000| North|
|  1|Rahul|  22|        IT| 40000| North|
|  4| Riya|NULL|   Finance| 45000| South|
| 10|Vikas|  40|     Sales| 90000|  NULL|
|  5| Amit|  28|        HR|  NULL|  East|
|  9| NULL|  26|        IT| 50000|  East|
+---+-----+----+----------+------+------+



In [10]:
df = df.fillna({"Age":25})

In [11]:
df = df.fillna({"Salary":0})

In [12]:
df = df.fillna({"Region":"Unknown"})

In [13]:
df = df.dropna(subset=["Name"])

In [14]:
df.filter(df.Age>25).show()

+---+-----+---+----------+------+-------+
| ID| Name|Age|Department|Salary| Region|
+---+-----+---+----------+------+-------+
|  3| Neha| 30|        IT| 60000|  North|
|  8|Priya| 35|   Finance| 80000|  South|
| 10|Vikas| 40|     Sales| 90000|Unknown|
|  5| Amit| 28|        HR|     0|   East|
+---+-----+---+----------+------+-------+



In [15]:
df.filter(
    (df.Age>=20)&
    (df.Age<=35)
).show()

+---+-----+---+----------+------+------+
| ID| Name|Age|Department|Salary|Region|
+---+-----+---+----------+------+------+
|  2| Aman| 25|        HR| 35000|  West|
|  3| Neha| 30|        IT| 60000| North|
|  8|Priya| 35|   Finance| 80000| South|
|  6|Rahul| 22|        IT| 40000| North|
|  1|Rahul| 22|        IT| 40000| North|
|  4| Riya| 25|   Finance| 45000| South|
|  5| Amit| 28|        HR|     0|  East|
+---+-----+---+----------+------+------+



In [16]:
df.filter(df.Department=="IT").show()

+---+-----+---+----------+------+------+
| ID| Name|Age|Department|Salary|Region|
+---+-----+---+----------+------+------+
|  3| Neha| 30|        IT| 60000| North|
|  6|Rahul| 22|        IT| 40000| North|
|  1|Rahul| 22|        IT| 40000| North|
+---+-----+---+----------+------+------+



In [17]:
df.filter(df.Region=="North").show()

+---+-----+---+----------+------+------+
| ID| Name|Age|Department|Salary|Region|
+---+-----+---+----------+------+------+
|  3| Neha| 30|        IT| 60000| North|
|  6|Rahul| 22|        IT| 40000| North|
|  1|Rahul| 22|        IT| 40000| North|
+---+-----+---+----------+------+------+



In [18]:
from pyspark.sql.functions import count

df.select(count("*")).show()

+--------+
|count(1)|
+--------+
|       9|
+--------+



In [19]:
from pyspark.sql.functions import avg

df.select(avg("Salary")).show()

+-----------------+
|      avg(Salary)|
+-----------------+
|46111.11111111111|
+-----------------+



In [20]:
from pyspark.sql.functions import max

df.select(max("Salary")).show()

+-----------+
|max(Salary)|
+-----------+
|      90000|
+-----------+



In [21]:
from pyspark.sql.functions import min

df.select(min("Salary")).show()

+-----------+
|min(Salary)|
+-----------+
|          0|
+-----------+



In [22]:
from pyspark.sql.functions import sum

df.select(sum("Salary")).show()

+-----------+
|sum(Salary)|
+-----------+
|     415000|
+-----------+



In [23]:
df.groupBy("Department") \
.agg(avg("Salary")) \
.show()

+----------+------------------+
|Department|       avg(Salary)|
+----------+------------------+
|     Sales|           57500.0|
|        HR|           17500.0|
|   Finance|           62500.0|
|        IT|46666.666666666664|
+----------+------------------+



In [25]:
df.groupBy("Department") \
.count() \
.show()

+----------+-----+
|Department|count|
+----------+-----+
|     Sales|    2|
|        HR|    2|
|   Finance|    2|
|        IT|    3|
+----------+-----+



In [26]:
df.groupBy("Department") \
.count() \
.filter("count>1") \
.show()

+----------+-----+
|Department|count|
+----------+-----+
|     Sales|    2|
|        HR|    2|
|   Finance|    2|
|        IT|    3|
+----------+-----+



In [27]:
df.groupBy("Department").count().show()

+----------+-----+
|Department|count|
+----------+-----+
|     Sales|    2|
|        HR|    2|
|   Finance|    2|
|        IT|    3|
+----------+-----+



In [28]:
df = df.withColumnRenamed("Salary","MonthlySalary")

In [29]:
from pyspark.sql.functions import col

df = df.withColumn(
    "Age",
    col("Age").cast("integer")
)

In [30]:
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Department: string (nullable = true)
 |-- MonthlySalary: integer (nullable = false)
 |-- Region: string (nullable = false)



In [31]:
from pyspark.sql.functions import avg

df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

df = df.dropDuplicates()

df = df.fillna({
    "Age":25,
    "Salary":0,
    "Region":"Unknown"
})

df = df.dropna(subset=["Name"])

result = (
    df.filter(df.Age>=20)
      .groupBy("Department")
      .agg(avg("Salary").alias("AverageSalary"))
)

result.show()

+----------+------------------+
|Department|     AverageSalary|
+----------+------------------+
|     Sales|           90000.0|
|        HR|           17500.0|
|   Finance|           62500.0|
|        IT|46666.666666666664|
+----------+------------------+

